# Multigrid Track — Phase 0 Validation Notebook

Reproduces every test of `multigrid_selfgravity_validation.pdf`: the shearing-box +
mesh-refinement infrastructure (orbital-advection toggle, uniform-boundary-level and annular
refinement policies, FARGO on refined meshes).

**Prerequisites**: the code built (`cmake .. && make -j8`; no FFT flag needed for this
notebook) and Julia packages `CairoMakie`, `IJulia`.

**Structure**
1. Helpers: running AthenaK, reading `.hst` and per-block `.bin`
2. Frame consistency: FARGO vs full-velocity (uniform grid)
3. Guard checks: the two policies must refuse illegal setups
4. FARGO on an annular refined ring vs its non-FARGO twin
5. The 20-orbit epicycle endurance test (the decisive one)
6. Two-level refinement (levels 1 and 2)
7. Mesh-layout visualization from `.bin` outputs
8. AMR: dynamic ring creation/destruction (Phase 0c) and its convergence
8. Playground

In [ ]:
const REPO   = expanduser("~/Library/CloudStorage/Dropbox/Research/code/athenak-multigrid")
const ATHENA = joinpath(REPO, "build", "src", "athena")
const INPUTS = joinpath(REPO, "inputs")
const RUN    = joinpath(REPO, "validation", "run")          # short shwave/ring tests
const RUNEPI = joinpath(REPO, "validation", "run_epicycle") # 20-orbit epicycle runs
mkpath(RUN); mkpath(RUNEPI)
isfile(ATHENA) || @warn "athena executable not found -- build first"
using CairoMakie, Printf
CairoMakie.activate!(type="png")
set_theme!(Theme(fontsize=13, Axis=(xgridcolor=(:gray, 0.25), ygridcolor=(:gray, 0.25))))
const C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100";


In [ ]:
"Run athena on `input` (path relative to inputs/) in `dir` with overrides; return stdout."
function run_athena(input; overrides=String[], dir=RUN)
    cmd = Cmd(String[ATHENA, "-i", joinpath(INPUTS, input), overrides...])
    read(pipeline(Cmd(cmd; dir=dir); stderr=devnull), String)
end

"Run athena expecting a FATAL abort; return the fatal message lines."
function run_expect_fatal(input; overrides=String[], dir=RUN)
    out = read(pipeline(ignorestatus(Cmd(Cmd(String[ATHENA, "-i",
              joinpath(INPUTS, input), overrides...]); dir=dir)); stderr=devnull), String)
    lines = split(out, '\n')
    i = findfirst(l -> occursin("FATAL", l), lines)
    i === nothing ? error("expected a FATAL error but the run proceeded!") :
                    join(lines[i:min(i+3, length(lines))], '\n')
end

"Read an AthenaK .hst file into a matrix (rows = outputs)."
function read_hst(f)
    rows = Float64[]; ncol = 0
    for ln in eachline(f)
        startswith(strip(ln), "#") && continue
        v = parse.(Float64, split(ln)); ncol = length(v); append!(rows, v)
    end
    permutedims(reshape(rows, ncol, :))
end

"Read an AthenaK .bin file into per-MeshBlock records (multilevel-safe)."
function read_bin_blocks(filename)
    open(filename, "r") do io
        startswith(readline(io), "Athena binary output") || error("not an AthenaK bin")
        npre = parse(Int, split(readline(io), "=")[end])
        ph = Dict(String(strip(k)) => String(strip(v)) for (k, v) in
                  (split(readline(io), "=") for _ in 1:npre-1))
        locsize = parse(Int, ph["size of location"])
        varsize = parse(Int, ph["size of variable"])
        nvars = parse(Int, split(readline(io), "=")[end])
        vars = String.(split(readline(io))[2:end])
        hsize = parse(Int, split(readline(io), "=")[end])
        header = String(read(io, hsize))
        m = match(r"nghost\s*=\s*(\d+)", header)
        ng = parse(Int, m.captures[1])
        locT = locsize == 8 ? Float64 : Float32
        varT = varsize == 8 ? Float64 : Float32
        blocks = NamedTuple[]
        while !eof(io)
            idx = Int.(reinterpret(Int32, read(io, 24))) .- ng
            n1 = idx[2]-idx[1]+1; n2 = idx[4]-idx[3]+1; n3 = idx[6]-idx[5]+1
            logical = Int.(reinterpret(Int32, read(io, 16)))       # lx1,lx2,lx3,level
            geom = Float64.(reinterpret(locT, read(io, 6*locsize)))  # x1min..x3max
            raw = reinterpret(varT, read(io, n1*n2*n3*nvars*varsize))
            data = reshape(Float64.(raw), (n1, n2, n3, nvars))
            push!(blocks, (logical=logical, geom=geom, data=data))
        end
        (blocks=blocks, vars=vars)
    end
end;


## 1. Frame consistency — FARGO vs full-velocity, uniform grid

`<shearing_box> orbital_advection = false` switches to the full-velocity frame: the
solver integrates the shear directly, with full Coriolis + tidal source terms and
momentum/energy offsets in the shear-periodic wrap. Same physics, different variables —
frame-independent diagnostics must agree to truncation order.

In [ ]:
for (bn, ov) in (("shw_fargo", String[]),
                 ("shw_nofargo", ["shearing_box/orbital_advection=false"]))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/hydro_incompress_shwave.athinput";
                   overrides=vcat("job/basename=$bn", ov))
end
a = read_hst(joinpath(RUN, "shw_fargo.hydro.hst"))
b = read_hst(joinpath(RUN, "shw_nofargo.hydro.hst"))
n = min(size(a, 1), size(b, 1))
@printf("mass : max rel diff = %.2e   (frame-independent, expect ~0)\n",
        maximum(abs.(a[1:n,3] .- b[1:n,3]))/maximum(abs.(a[1:n,3])))
@printf("x-KE : max rel diff = %.4f   (truncation gap between the two schemes)\n",
        maximum(abs.(a[1:n,7] .- b[1:n,7]))/maximum(abs.(a[1:n,7])))


## 2. Guard checks — the policies must refuse illegal setups

The policies are enforced with fatal errors, not conventions. Three runs that must
abort: a refined region that puts *mixed* levels on the shear-periodic $x_1$ boundary,
refining only *one* boundary face (mixed levels across the wrap), and a partial-$x_2$
ring with FARGO on.

In [ ]:
println("--- uniform-boundary-level policy (partial boundary refinement):")
println(run_expect_fatal("shearing_box/hydro_incompress_shwave_smr.athinput";
        overrides=["refined_region1/x1min=-0.24", "refined_region1/x1max=-0.13"]))
println("\n--- uniform-boundary-level policy (only ONE face refined):")
println(run_expect_fatal("shearing_box/hydro_incompress_shwave_smr_bndry.athinput";
        overrides=["refined_region2/x1min=0.0", "refined_region2/x1max=0.12"]))
println("\n--- annular policy (partial ring + FARGO):")
println(run_expect_fatal("shearing_box/hydro_incompress_shwave_smr.athinput";
        overrides=["shearing_box/orbital_advection=true"]))


## 3. FARGO on an annular refined ring

The `_ring` input refines a full-$x_2$ level-1 annulus (interior in $x_1$). Every
$x_2$-face neighbor is then same-level, so the per-block shift kernel and the existing
communication run unchanged, level by level. Checks: exact mass conservation, and
agreement with the non-FARGO twin at the same truncation level as on uniform grids.

In [ ]:
for (bn, ov) in (("ring_fargo", String[]),
                 ("ring_nofargo", ["shearing_box/orbital_advection=false"]))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/hydro_incompress_shwave_smr_ring.athinput";
                   overrides=vcat("job/basename=$bn", ov))
end
rf = read_hst(joinpath(RUN, "ring_fargo.hydro.hst"))
rn = read_hst(joinpath(RUN, "ring_nofargo.hydro.hst"))
n = min(size(rf, 1), size(rn, 1))
@printf("ring+FARGO mass drift        = %.2e   (expect exactly 0)\n",
        maximum(abs.(rf[:,3] .- rf[1,3])))
@printf("ring FARGO vs non-FARGO x-KE = %.4f   (uniform-grid gap was 3.3%%)\n",
        maximum(abs.(rf[1:n,7] .- rn[1:n,7]))/maximum(abs.(rf[1:n,7])))


## 4. Refined $x_1$ boundary faces

The uniform-boundary-level rule permits refinement *at* the shear boundary when
**both** faces are refined uniformly (full $x_2$/$x_3$, same level) — natural for a
structure at the boundary, whose shear-periodic images live on both faces. The
`_bndry` input refines the leftmost and rightmost root columns to level 1. The
decisive check is again the spatially uniform epicycle: it must pass through the
level-1 shear-periodic wrap and the level-1 azimuthal redistribution exactly.

In [ ]:
for (bn, ov) in (("shwave2_smr_bndry", String[]),
                 ("epi_bndry", ["problem/ipert=1"]))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/hydro_incompress_shwave_smr_bndry.athinput";
                   overrides=vcat("job/basename=$bn", ov))
end
eb = read_hst(joinpath(RUN, "epi_bndry.hydro.hst"))
t = eb[:, 1]; amp = 1.0e-4; V = 0.125
ana = 0.5*V*amp^2 .* cos.(t).^2
@printf("epicycle, boundary annuli + FARGO vs analytic: max rel = %.3e\n",
        maximum(abs.(eb[:,7] .- ana))/maximum(ana))
sb = read_hst(joinpath(RUN, "shwave2_smr_bndry.hydro.hst"))
su = read_hst(joinpath(RUN, "shw_fargo.hydro.hst"))     # uniform reference (Sec. 1)
n = min(size(sb, 1), size(su, 1))
@printf("shwave, boundary annuli: mass drift = %.2e,  x-KE vs uniform = %.4f\n",
        maximum(abs.(sb[:,3] .- sb[1,3])),
        maximum(abs.(sb[1:n,7] .- su[1:n,7]))/maximum(abs.(su[1:n,7])))


## 5. The endurance test — epicycles for 20 orbits

`ipert=1` initializes a spatially **uniform** epicycle $v_x(0)=A$, an exact nonlinear
solution: $v_x = A\cos\kappa t$, $v_y' = -(A\kappa/2\Omega)\sin\kappa t$, with
$\kappa=\Omega$ at $q=3/2$. A uniform field must pass through the $y$-shift *exactly*,
so any indexing/communication defect at ring or level boundaries appears as $O(1)$
corruption — while phase/amplitude errors of the time integrator stay tiny and smooth.
The **amplitude invariant** $A(t)=\sqrt{v_x^2+(2\Omega/\kappa)^2v_y'^2}$ separates
numerical damping (drift in $A$) from phase error.

Note on units: `amp = 0.1` is a velocity in code units (`shwave.cpp` never multiplies
by $c_s$); with the ideal EOS at $p_0=\rho_0=1$, $c_s=\sqrt{5/3}$, so this epicycle is
$0.077\,c_s$ — the plot is normalized to `amp`, oscillating between $\pm1$.

In [ ]:
# ~1 min (uniform) + ~3 min each (ring, bndry) if the runs don't exist yet
isfile(joinpath(RUNEPI, "epi_unif.hydro.hst")) ||
    run_athena("shearing_box/epicycle.athinput"; dir=RUNEPI)
isfile(joinpath(RUNEPI, "epi_ring.hydro.hst")) ||
    run_athena("shearing_box/epicycle_smr_ring.athinput"; dir=RUNEPI)
isfile(joinpath(RUNEPI, "epi_bndry.hydro.hst")) ||
    run_athena("shearing_box/epicycle_smr_bndry.athinput"; dir=RUNEPI)

amp, Ω, q = 0.1, 1.0, 1.5
κ = sqrt(2*(2-q))*Ω
Torb = 2π/Ω
u = read_hst(joinpath(RUNEPI, "epi_unif.hydro.hst"))
r = read_hst(joinpath(RUNEPI, "epi_ring.hydro.hst"))
bd = read_hst(joinpath(RUNEPI, "epi_bndry.hydro.hst"))
vx_u = u[:,4]./u[:,3]; vy_u = u[:,5]./u[:,3]; t_u = u[:,1]
vx_r = r[:,4]./r[:,3]; vy_r = r[:,5]./r[:,3]; t_r = r[:,1]
vx_b = bd[:,4]./bd[:,3]; vy_b = bd[:,5]./bd[:,3]; t_b = bd[:,1]
envel(vx, vy) = sqrt.(vx.^2 .+ (2Ω/κ)^2 .* vy.^2)
A_u, A_r, A_b = envel(vx_u, vy_u), envel(vx_r, vy_r), envel(vx_b, vy_b)
ana(t) = amp .* cos.(κ .* t)
for (tag, t, v, A) in (("uniform    ", t_u, vx_u, A_u), ("ring+FARGO ", t_r, vx_r, A_r),
                       ("bndry+FARGO", t_b, vx_b, A_b))
    @printf("%s  vx∈[%+.5f, %+.5f]  amplitude drift %+.3f%%  max|vx-analytic|/amp = %.4f\n",
            tag, minimum(v), maximum(v), 100*(A[end]-A[1])/amp,
            maximum(abs.(v .- ana(t)))/amp)
end
n = min(size(r,1), size(bd,1))
@printf("\nboundary-annuli vs interior-ring endurance run, all columns: max diff = %.1e\n",
        maximum(abs.(r[1:n,:] .- bd[1:n,:])))


In [ ]:
fig = Figure(size=(880, 620))
ax1 = Axis(fig[1,1]; xlabel="t / T_orb", ylabel="vₓ / amp",
           title="Epicyclic radial velocity over 20 orbits")
hlines!(ax1, [-1, 1]; color=(:gray, 0.55), linestyle=:dash)
lines!(ax1, t_u ./ Torb, vx_u ./ amp; color=C1, linewidth=1.2, label="uniform grid")
lines!(ax1, t_r ./ Torb, vx_r ./ amp; color=C3, linewidth=1.2, linestyle=:dash,
       label="annular ring + FARGO")
lines!(ax1, t_b ./ Torb, vx_b ./ amp; color=C4, linewidth=1.2, linestyle=:dot,
       label="boundary annuli + FARGO")
axislegend(ax1; position=:rt, framevisible=false, orientation=:horizontal)
ylims!(ax1, -1.35, 1.35)

sel_u = t_u ./ Torb .>= 18; sel_r = t_r ./ Torb .>= 18
tf = range(18Torb, t_u[end]; length=600)
ax2 = Axis(fig[2,1]; xlabel="t / T_orb", ylabel="vₓ / amp", height=150,
           title="last two orbits vs analytic amp·cos(κt)")
lines!(ax2, tf ./ Torb, ana(tf) ./ amp; color=C2, linewidth=2.5, label="analytic")
scatter!(ax2, t_u[sel_u] ./ Torb, vx_u[sel_u] ./ amp; color=C1, markersize=6,
         label="uniform")
scatter!(ax2, t_r[sel_r] ./ Torb, vx_r[sel_r] ./ amp; color=C3, marker=:rect,
         markersize=5, label="ring")
sel_b = t_b ./ Torb .>= 18
scatter!(ax2, t_b[sel_b] ./ Torb, vx_b[sel_b] ./ amp; color=C4, marker=:diamond,
         markersize=5, label="bndry")
axislegend(ax2; position=:rt, framevisible=false, orientation=:horizontal)

ax3 = Axis(fig[3,1]; xlabel="t / T_orb", ylabel="A(t) / amp", height=150,
           title="amplitude invariant — flat means no numerical damping")
hlines!(ax3, [1.0]; color=(:gray, 0.55), linestyle=:dash)
lines!(ax3, t_u ./ Torb, A_u ./ amp; color=C1, linewidth=1.5, label="uniform")
lines!(ax3, t_r ./ Torb, A_r ./ amp; color=C3, linewidth=1.5, linestyle=:dash,
       label="ring + FARGO")
lines!(ax3, t_b ./ Torb, A_b ./ amp; color=C4, linewidth=1.5, linestyle=:dot,
       label="bndry + FARGO")
axislegend(ax3; position=:rb, framevisible=false, orientation=:horizontal)
fig


## 6. Two-level refinement (levels 1 and 2)

Both policies recurse in level, so deeper hierarchies need no new machinery — verified
with two configurations at levels 1+2 (8 root + 32 level-1 + 256 level-2 blocks each,
FARGO on): nested interior rings, and level-2 boundary annuli with level-1 buffer rings
inboard (2:1 nesting). The epicycle is again the decisive check.

In [ ]:
for (inp, bn) in (("hydro_incompress_shwave_smr_ring2.athinput", "epi_ring2"),
                  ("hydro_incompress_shwave_smr_bndry2.athinput", "epi_bndry2"))
    isfile(joinpath(RUN, bn * ".hydro.hst")) ||
        run_athena("shearing_box/" * inp;
                   overrides=["job/basename=$bn", "problem/ipert=1"])
end
e2r = read_hst(joinpath(RUN, "epi_ring2.hydro.hst"))
e2b = read_hst(joinpath(RUN, "epi_bndry2.hydro.hst"))
t2 = e2r[:,1]; amp2 = 1.0e-4; V = 0.125
ana2 = 0.5*V*amp2^2 .* cos.(t2).^2
@printf("epicycle, nested rings (L1+L2)      vs analytic: max rel = %.3e\n",
        maximum(abs.(e2r[:,7] .- ana2))/maximum(ana2))
@printf("epicycle, two-level boundary annuli vs analytic: max rel = %.3e\n",
        maximum(abs.(e2b[:,7] .- ana2))/maximum(ana2))
n = min(size(e2r,1), size(e2b,1))
@printf("the two two-level meshes vs each other, all columns: max diff = %.1e\n",
        maximum(abs.(e2r[1:n,:] .- e2b[1:n,:])))


## 7. Mesh-layout visualization

Drawn directly from the per-block structure of the `.bin` outputs — the same reader you
will use for any refined-mesh analysis. Left: the interior patch (legal only without
FARGO). Right: the annular ring (FARGO-legal). Orange lines mark the shear-periodic
$x_1$ boundaries that refinement must avoid.

In [ ]:
function draw_blocks!(ax, fd; title="")
    lev_col = Dict(0 => :white, 1 => "#dbe9fb", 2 => "#b9d5f6")
    z0 = minimum(b.geom[5] for b in fd.blocks)
    for b in fd.blocks
        b.geom[5] ≈ z0 || continue          # one z-layer
        g = b.geom; lev = b.logical[4]
        poly!(ax, Rect2(g[1], g[3], g[2]-g[1], g[4]-g[3]);
              color=get(lev_col, lev, "#8fbdf0"), strokecolor=:black, strokewidth=0.8)
    end
    ax.title = title
end

# the patch (partial-x2) run is legal only with FARGO off; generate it if absent
isfile(joinpath(RUN, "shwave2_nofargo_smr.hydro.hst")) ||
    run_athena("shearing_box/hydro_incompress_shwave_smr.athinput";
               overrides=["shearing_box/orbital_advection=false",
                          "job/basename=shwave2_nofargo_smr"])
f_patch = first(sort(filter(f -> occursin("shwave2_nofargo_smr.hydro_w", f),
                            readdir(joinpath(RUN, "bin"); join=true))))
f_ring = first(sort(filter(f -> occursin("ring_fargo.hydro_w", f),
                           readdir(joinpath(RUN, "bin"); join=true))))
fig = Figure(size=(860, 430))
for (i, (f, ttl)) in enumerate(((f_patch, "patch (FARGO off only)"),
                                (f_ring, "annular ring (FARGO OK)")))
    ax = Axis(fig[1, i]; xlabel="x", ylabel=(i == 1 ? "y" : ""), aspect=DataAspect())
    draw_blocks!(ax, read_bin_blocks(f); title=ttl)
    vlines!(ax, [-0.25, 0.25]; color=C2, linewidth=3)
end
fig


## 8. AMR — dynamic ring creation and destruction (Phase 0c)

AMR renumbers MeshBlock GIDs on every update, so the shear-boundary communication
state (GID lists, MPI requests, buffers) built at construction goes stale.
`ShearingBox::ReinitAfterMeshUpdate` (commit `b5388c02`) rebuilds it after every
mesh update; a **graded cap** in `CheckForRefinement` refuses refinement whose 2:1
balancing would reach the shear-periodic boundary columns
($2^{s+1}-1 \le l_{x1} \le n_{x1}-2^{s+1}$ at $s$ levels above the boundary
level); and an upstream derefinement-sort bug (end iterator excluded the last
element → order-dependent partial ring merges, commit `f0acd383`) is fixed.

The dynamic exercise is a **prescribed moving ring** (`MovingRingRefine` in the
shwave pgen): refine blocks overlapping $[x_c(t)-w, x_c(t)+w]$ with
$x_c = A\sin(ft)$, derefine everything else. With $A$ past the vetoed boundary
columns, all rings are destroyed near each turning point and re-created on the way
back. The 20-orbit epicycle run creates 2296 / destroys 2240 MeshBlocks with
exactly zero mass drift, bit-reproducible across reruns.

In [ ]:
# ~4 min (20-orbit AMR) + ~2 min (two-level) + ~2 min (non-FARGO) if absent
for (hst, inp, log) in (("epi_amr.hydro.hst", "epicycle_amr_ring.athinput", "log_amr.txt"),
                        ("epi_amr2.hydro.hst", "epicycle_amr_ring2.athinput", "log_amr2.txt"),
                        ("epi_amr_nofargo.hydro.hst", "epicycle_amr_ring_nofargo.athinput",
                         "log_amr_nofargo.txt"))
    if !isfile(joinpath(RUNEPI, hst))
        out = run_athena("shearing_box/" * inp; dir=RUNEPI)
        write(joinpath(RUNEPI, log), out)
    end
end
for log in ("log_amr.txt", "log_amr2.txt", "log_amr_nofargo.txt")
    f = joinpath(RUNEPI, log)
    isfile(f) && for ln in eachline(f)
        occursin("created", ln) && @printf("%-18s %s\n", log, ln)
    end
end

# amplitude drift and time-matched x-KE difference vs the section-5 references
amp, Ω, q = 0.1, 1.0, 1.5
κ = sqrt(2*(2-q))*Ω
u = read_hst(joinpath(RUNEPI, "epi_unif.hydro.hst"))
r = read_hst(joinpath(RUNEPI, "epi_ring.hydro.hst"))
a = read_hst(joinpath(RUNEPI, "epi_amr.hydro.hst"))
envel(h) = sqrt.((h[:,4]./h[:,3]).^2 .+ (2Ω/κ)^2 .* (h[:,5]./h[:,3]).^2)
for (tag, h) in (("uniform ", u), ("ring    ", r), ("amr     ", a))
    A = envel(h)
    @printf("%s mass drift %.1e   amplitude drift %+.3e\n", tag,
            maximum(abs.(h[:,3] .- h[1,3]))/h[1,3], (A[end]-A[1])/amp)
end
# linear-interp time-matched max |Δ(1-KE)| (col 8, ideal EOS), scaled to the peak
function tmdiff(p, b; col=8)
    mx = 0.0
    for i in 1:size(p,1)
        t = p[i,1]; j = searchsortedlast(b[:,1], t)
        (j < 1 || j >= size(b,1)) && continue
        w = (t - b[j,1])/(b[j+1,1] - b[j,1])
        mx = max(mx, abs(p[i,col] - ((1-w)*b[j,col] + w*b[j+1,col])))
    end
    mx / maximum(b[:,col])
end
@printf("\ntime-matched x-KE:  amr vs ring %.2e   amr vs uniform %.2e   ring vs uniform %.2e\n",
        tmdiff(a, r), tmdiff(a, u), tmdiff(r, u))

In [ ]:
# MeshBlock-count history from the bin outputs, with the prescribed ring center
files = sort(filter(f -> startswith(f, "epi_amr.hydro_w."),
                    readdir(joinpath(RUNEPI, "bin"))))
nmb = [length(read_bin_blocks(joinpath(RUNEPI, "bin", f)).blocks) for f in files]
tt = 0.628 .* (0:length(nmb)-1)   # output cadence dt = 0.628 (see the input file)
Torb = 2π
fig = Figure(size=(880, 300))
ax = Axis(fig[1,1]; xlabel="t / T_orb", ylabel="MeshBlocks",
          title="moving-ring AMR: block count (blue) and ring center (orange)")
stairs!(ax, collect(tt) ./ Torb, nmb; color=C1, step=:post)
axr = Axis(fig[1,1]; yaxisposition=:right, ylabel="x_c(t)", ylabelcolor=C2,
           yticklabelcolor=C2)
hidespines!(axr); hidexdecorations!(axr)
lines!(axr, collect(tt) ./ Torb, 7.0 .* sin.(tt); color=(C2, 0.55))
hlines!(axr, [-5, 5]; color=(:gray, 0.5), linestyle=:dash)
xlims!(ax, 0, 20); xlims!(axr, 0, 20)
@printf("blocks: min %d  max %d  (16 = all-root)\n", minimum(nmb), maximum(nmb))
fig

### 8.1 Wave structure through a moving refinement boundary

The epicycle is spatially uniform — it never exercises prolongation/restriction on
real structure. The vortical-shwave AMR test runs the JG05 problem of section 1
under a moving ring ($x_c = 0.2\sin t$, $w = 0.05$), so wave structure crosses
every regrid event. At $64^2$ the moving ring roughly doubles the static-ring
truncation (the ring covers only ~1.6 wavelengths and full destruction restricts
fine data mid-swing); at $128^2$ moving and static rings agree to $1.8\times10^{-4}$
— the regridding cost converges away with no floor.

In [ ]:
# ~2 min (64²) + ~15 min for the 128² trio if absent
isfile(joinpath(RUN, "shwave2_amr.hydro.hst")) || begin
    out = run_athena("shearing_box/hydro_incompress_shwave_amr_ring.athinput"; dir=RUN)
    write(joinpath(RUN, "log_shwave2_amr.txt"), out)
end
r128 = [("shw_fargo128", "hydro_incompress_shwave.athinput", String[]),
        ("ring_fargo128", "hydro_incompress_shwave_smr_ring.athinput", String[]),
        ("shwave2_amr128", "hydro_incompress_shwave_amr_ring.athinput",
         ["mesh_refinement/max_nmb_per_rank=384"])]
for (bn, inp, extra) in r128
    isfile(joinpath(RUN, bn * ".hydro.hst")) && continue
    out = run_athena("shearing_box/" * inp;
                     overrides=vcat(["mesh/nx1=128", "mesh/nx2=128",
                                     "job/basename=" * bn], extra), dir=RUN)
    write(joinpath(RUN, "log_" * bn * ".txt"), out)
end

# keep the last monotone-time segment (guards against appended reruns)
seg(h) = begin
    s = 1
    for i in 2:size(h,1); h[i,1] <= h[i-1,1] && (s = i); end
    h[s:end, :]
end
hst(f) = seg(read_hst(joinpath(RUN, f)))
for (res, fu, fr, fa) in (("64 ", "shw_fargo.hydro.hst", "ring_fargo.hydro.hst",
                           "shwave2_amr.hydro.hst"),
                          ("128", "shw_fargo128.hydro.hst", "ring_fargo128.hydro.hst",
                           "shwave2_amr128.hydro.hst"))
    hu, hr, ha = hst(fu), hst(fr), hst(fa)
    pu = maximum(hu[:,7])   # isothermal EOS: x-KE is column 7
    @printf("N=%s  peak x-KE err: ring %+.2f%%  amr %+.2f%%   amr-vs-ring %.1e   mass %.1e\n",
            res, 100*(maximum(hr[:,7])/pu - 1), 100*(maximum(ha[:,7])/pu - 1),
            tmdiff(ha, hr; col=7), maximum(abs.(ha[:,3] .- ha[1,3]))/ha[1,3])
end

In [ ]:
hu, hr, ha = hst("shw_fargo128.hydro.hst"), hst("ring_fargo128.hydro.hst"),
             hst("shwave2_amr128.hydro.hst")
fig = Figure(size=(700, 340))
ax = Axis(fig[1,1]; xlabel="t", ylabel="x-kinetic energy",
          title="128² vortical shwave through the swing (t_swing = 2.67)")
lines!(ax, hu[:,1], hu[:,7]; color=C1, linewidth=4, label="uniform")
lines!(ax, hr[:,1], hr[:,7]; color=C3, linewidth=2.2, label="static ring (SMR)")
lines!(ax, ha[:,1], ha[:,7]; color=C2, linewidth=1.2, linestyle=:dash,
       label="moving ring (AMR)")
axislegend(ax; position=:lt, framevisible=false)
fig

## Playground

| override | effect |
|---|---|
| `shearing_box/orbital_advection=false` | full-velocity (non-FARGO) mode |
| `problem/ipert=1|2|3` | epicycle / vortical shwave / compressive shwave |
| `refined_region1/x1min=…` etc. | move/resize the refined region (policies enforced) |
| `problem/amp=…` | perturbation amplitude (code units, not $c_s$) |
| `time/tlim=…` | run length |

**Next (Phase 1):** `shear_periodic` boundary conditions in the multigrid driver,
validated against the FFT oracle — see `multigrid_selfgravity_validation.pdf` Sec. 5.